In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass, InitVar, field, asdict

### DATASET and PARAMETER

@dataclass(frozen=True)
class HyperParameter:
    batch_size: int
    num_epochs: int
    learning_rate: float

@dataclass
class MNIST:
    root: InitVar[str]
    param: InitVar[HyperParameter]

    train: DataLoader = field(init=False)
    test:  DataLoader = field(init=False)
    total: DataLoader = field(init=False)

    def __post_init__(self, root, param):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        train_set = datasets.MNIST(
            root=root,
            train=True,
            download=True,
            transform=transform,
        )
        test_set = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=transform,
        )
        self.train = DataLoader(
            dataset=train_set,
            batch_size=param.batch_size, shuffle=True
        )
        self.test = DataLoader(
            dataset=test_set,
            batch_size=param.batch_size, shuffle=False
        )
        self.total = DataLoader(
            dataset=train_set+test_set,
            batch_size=param.batch_size, shuffle=False
        )

In [18]:
### NEURAL NETWORK DEFINITION
class MNISTSLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 10, bias=True)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [19]:
import os
from pathlib import Path
from datetime import datetime

### TRAIN / EVALUATION ROUTINE

@dataclass
class Routine:
    device: object
    criterion: object
    optimizer: object

    def _routine(self, model, loader):
        num_hit, running_loss = 0, 0.0
        
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            loss = self.criterion(outputs, labels)

            if torch.is_grad_enabled():
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

            num_hit += (predicted==labels).sum().item()
            running_loss += loss.item()
            
        return num_hit / len(loader.dataset), \
               running_loss / len(loader)

    def train(self, model, loader):
        model.train()
        return self._routine(model, loader)

    def test(self, model, loader):
        model.eval()
        with torch.no_grad():
            return self._routine(model, loader)
            
@dataclass
class Milestone:
    parent: str | Path
    model_name: str
    ext: str
    now: str = field(init=False)

    def __post_init__(self):
        self.now = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
        self.parent = Path(self.parent)

        if not self.parent.exists():
            self.parent.mkdir()

    def __iter__(self):
        files = self.parent.glob(f"{self.model_name}-*.{self.ext}")
        return iter(sorted(files, key=lambda f: f.name, reverse=True))

    def save(self, **kwargs):
        torch.save(kwargs, self.parent / f"{self.model_name}-{self.now}.{self.ext}")

In [39]:
### TRAIN / EVALUATION ROUTINE
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
param = HyperParameter(
    batch_size=128,
    num_epochs=20,
    learning_rate=1e-3
)
mnist = MNIST('./data', param)
model = MNISTSLP().to(device)

criterion =  nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=param.learning_rate)
routine = Routine(device, criterion, optimizer)

In [40]:
milestone = Milestone("./milestones", type(model).__name__, 'pth')
print(f"Train Starts at: {milestone.now}")
print(f"Target Model: {milestone.model_name}")
print('')
print(f"| {'EPOCH': ^5} | {'TRAIN': ^33} | {'TEST': ^33} |")
print(f"| {'     ': ^5} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} "
      f"| {'ACCURACY': ^15} | {'LOSS': ^15} |")

checkpoint = None
for epoch in range(param.num_epochs):
    train_acc, train_loss = routine.train(model, mnist.train)
    test_acc, test_loss = routine.test(model, mnist.test)

    print(f"| {epoch: ^5} "
          f"| {train_acc: ^15.2%} | {train_loss: ^15.5} "
          f"| {test_acc: ^15.2%} | {test_loss: ^15.5} |", end='')

    if (checkpoint is None) or (checkpoint['accuracy'] < test_acc):
        checkpoint = dict(
            epoch=epoch,
            accuracy=test_acc,
            param=asdict(param),
            model=model.state_dict(),
            optim=optimizer.state_dict(),
        )
        milestone.save(**checkpoint)
        print(' *', end='')
    print('')

Train Starts at: 2026-04-24-20-26-42
Target Model: MNISTSLP

| EPOCH |               TRAIN               |               TEST                |
|       |    ACCURACY     |      LOSS       |    ACCURACY     |      LOSS       |
|   0   |     85.81%      |     0.52591     |     90.67%      |     0.33046     | *
|   1   |     90.34%      |     0.33695     |     91.22%      |     0.30325     | *
|   2   |     90.93%      |     0.31375     |     91.57%      |     0.29456     | *
|   3   |     91.30%      |     0.30294     |     91.79%      |     0.28822     | *
|   4   |     91.63%      |     0.2956      |     91.42%      |     0.29819     |
|   5   |     91.76%      |     0.29047     |     91.81%      |     0.28506     | *
|   6   |     91.87%      |     0.28682     |     92.01%      |     0.28149     | *
|   7   |     91.97%      |     0.28289     |     91.46%      |     0.29239     |
|   8   |     92.10%      |     0.28116     |     91.81%      |     0.28369     |
|   9   |     92.02%     

In [41]:
model_name = 'MNISTSLP'
test_ms = Milestone(milestone.parent, model_name, milestone.ext)
test_model = globals()[model_name]()
test_model.eval()

with torch.no_grad():
    for ms_file in test_ms:
        checkpoint = torch.load(ms_file)
    
        test_model.load_state_dict(checkpoint['model'])
        test_model.to(device)
        accuracy, _ = routine.test(test_model, mnist.total)
        print(ms_file.stem, ':\t', f"{accuracy: .2%}")

MNISTSLP-2026-04-24-20-26-42 :	  92.62%
MNISTSLP-2026-04-24-20-18-38 :	  91.55%
MNISTSLP-2026-04-24-16-27-39 :	  92.00%
MNISTSLP-2026-04-24-14-11-25 :	  91.57%
MNISTSLP-2026-04-24-14-05-59 :	  92.60%
MNISTSLP-2026-04-24-13-57-13 :	  92.51%
MNISTSLP-2026-04-24-13-54-36 :	  92.38%
MNISTSLP-2026-04-24-13-52-29 :	  91.39%
